In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexerClient
from azure.search.documents.indexes.models import (
    SearchIndexer,
    FieldMapping,
    IndexingParameters
)

In [ ]:
indexer_client = SearchIndexerClient(
    endpoint="https://rush-lyric-ai-search.search.windows.net", 
    credential=AzureKeyCredential("<REDACTED_API_KEY>")
)

In [17]:
# 2a. Configure Indexing Parameters (CRITICAL for JSON/JSONL)
# This tells the indexer to treat each line as a separate document.
parameters = IndexingParameters(configuration={"parsingMode": "jsonLines"})

In [18]:
# 2b. Create the Indexer
indexer = SearchIndexer(
    name="rush-lyrics-indexer",
    data_source_name="rush-lyrics-ds", 
    target_index_name="integrated-index",
    skillset_name="rush-lyrics-skillset",
    field_mappings=[
        FieldMapping(source_field_name="id", target_field_name="id")
    ]
)

In [19]:
indexer_client.create_indexer(indexer)

HttpResponseError: () Data source does not contain column 'id', which is required because it maps to the document key field 'id' in the index 'integrated-index'. Ensure that the 'id' column is present in the data source, or add a field mapping that maps one of the existing column names to 'id'.
Code: 
Message: Data source does not contain column 'id', which is required because it maps to the document key field 'id' in the index 'integrated-index'. Ensure that the 'id' column is present in the data source, or add a field mapping that maps one of the existing column names to 'id'.

In [20]:
from azure.search.documents.indexes.models import IndexingParameters, SearchIndexer, FieldMapping
# 1. Define the "JSON Lines" parameter
# This is the missing piece that forces Azure to recognize the 'id' field
params = IndexingParameters(configuration={"parsingMode": "jsonLines"})
# 2. Re-define the indexer object with these parameters
indexer = SearchIndexer(
    name="rush-lyrics-indexer",
    data_source_name="rush-lyrics-ds", 
    target_index_name="integrated-index",
    skillset_name="rush-lyrics-skillset",
    parameters=params,  # <--- CRITICAL: Do not omit this
    field_mappings=[
        FieldMapping(source_field_name="id", target_field_name="id")
    ]
)
# 3. Create or Update to apply the changes
indexer_client.create_or_update_indexer(indexer)
print("Indexer successfully updated with parsingMode: jsonLines")

Indexer successfully updated with parsingMode: jsonLines


In [22]:
# Updated for the latest azure-search-documents SDK
indexer_status = indexer_client.get_indexer_status("rush-lyrics-indexer")

print(f"Indexer Service Status: {indexer_status.status}")

# Check the most recent execution result
last_result = indexer_status.last_result

if last_result:
    print(f"Last Run Status: {last_result.status}")
    # Use 'item_count' and 'failed_item_count' for the latest SDK
    print(f"Items Processed: {last_result.item_count}")
    print(f"Items Failed: {last_result.failed_item_count}")
    
    if last_result.errors:
        print(f"\nErrors: {len(last_result.errors)}")
else:
    print("No run history found.")

Indexer Service Status: running
Last Run Status: success
Items Processed: 150
Items Failed: 0


In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexerClient
from azure.search.documents.indexes.models import SearchIndexer, FieldMapping, IndexingParameters

# 1. Credentials Setup
# Using the keys from your notes (Step 6.0.1)
endpoint = "https://rush-lyric-ai-search.search.windows.net"
admin_key = "<REDACTED_API_KEY>"

# 2. Initialize the Client
indexer_client = SearchIndexerClient(
    endpoint=endpoint, 
    credential=AzureKeyCredential(admin_key)
)

# 3. Define Indexing Parameters for JSONL
parameters = IndexingParameters(configuration={"parsingMode": "jsonLines"})

# 4. Define the Indexer with Output Field Mappings
indexer = SearchIndexer(
    name="rush-lyrics-indexer",
    data_source_name="rush-lyrics-ds", 
    target_index_name="integrated-index",
    skillset_name="rush-lyrics-skillset",
    parameters=parameters,
    field_mappings=[
        FieldMapping(source_field_name="id", target_field_name="id")
    ],
    output_field_mappings=[
        # Maps the Skillset vector output to the Index field
        FieldMapping(source_field_name="/document/vector", target_field_name="vector")
    ]
)

# 5. Apply the Update and Run
indexer_client.create_or_update_indexer(indexer)
indexer_client.run_indexer("rush-lyrics-indexer")

print("Indexer client initialized and indexer update triggered successfully.")